# 1.- S02 | Practica guiada: del DataFrame a las primeras evidencias

**Ciencia de Datos Aplicada a Negocio**  
Master Analitica de Negocios, UNIE | Edicion 2604  
Jaime Pineda | 11/05/2026 | v0 (notebook inicial)

Este notebook acompana la S02. Convierte las preguntas de S01 (Canvas v1, baseline 20.2%, tres hipotesis PECO) en evidencia tabular real sobre 110.527 citas. El objetivo no es memorizar Pandas sino aprender a recorrer el flujo CRISP-DM > EDA > DAMA-5 > PECO con codigo explicito.

**Fuentes canonicas** (no duplicar contenido aqui):

- Guia docente: `S02_2604_guia_clase.md` (v3)
- Matriz operativa: `Curso/2604/matriz_maestra_S02_S04.md` (v3)
- Marco transversal: `Curso/2604/MARCO_ANALITICO_TRANSVERSAL.md`
- Slides proyectadas en clase: `S02_2604_slides.md`

## 1.1.- Como usar este notebook

Tres reglas de uso:

1. **Una decision cognitiva por celda.** Cada celda responde una pregunta: que calculo, que metodo uso, que espero ver, como lo leo.
2. **Variables intermedias antes que encadenamiento.** Primero nombramos los pasos (mascara, conversion, calculo) y solo al final, si conviene, los compactamos.
3. **Lectura despues de cada bloque.** El codigo produce numeros; la lectura los convierte en negocio. Sin lectura, el codigo es ruido.

**Convenciones**

- `random_state = 14` en cualquier muestreo o split del curso.
- Codigo y nombres de variables en ingles; titulos, narrativa y lecturas en espanol.
- Marcos transversales nombrados con palabra ancla: `CRISP-DM: Data Understanding`, `EDA: forma`, `DAMA-5: validez`, `PECO: P/E/C/O`, `Metodos Pandas: <familia>`.

## 1.2.- Objetivos de la sesion

Al terminar S02 debes poder:

1. Cargar el CSV de No-Show con `pd.read_csv` y reconocer un `DataFrame` como objeto con atributos y metodos.
2. Recorrer las 5 preguntas canonicas de EDA sobre el dataset.
3. Distinguir `Series` y `DataFrame` y aplicar `value_counts`, `.mean()`, mascaras booleanas y `.loc` con criterio.
4. Construir variables derivadas (`NoShow`, `WaitingDays`, `AppointmentWeekday`) en pasos explicitos.
5. Combinar mascaras con `&` para producir `df_analysis`.
6. Generar 3 evidencias por `groupby` simple (H1, H2, H3 PECO) con `mean` y `count` juntos.
7. Actualizar el Project Canvas (casillas 3, 4 y 5) con conteos reales.

# 2.- B0 Apertura: marcos y mapa de la sesion

> Lente principal: `CRISP-DM: Data Understanding`.

Esta seccion no tiene codigo. Es el momento conceptual de la sesion: recap de S01, ubicacion en CRISP-DM y declaracion de los marcos transversales que vamos a usar.

## 2.1.- Que traemos de S01

- **CRISP-DM** localizado: transicion de Business Understanding a Data Understanding.
- **Baseline** establecido: tasa de no-show 20.2% (22.319 sobre 110.527).
- **Tres hipotesis PECO** sin verificar todavia:
  - H1: esperas largas (>14 dias) se asocian con mayor no-show.
  - H2: el dia de la semana influye en la tasa.
  - H3: el SMS reduce el no-show (o no).
- **Project Canvas v1** con casillas 1, 2, 4, 5 y 6 en borrador.

## 2.2.- Roadmap CRISP-DM del curso

Hoy (S02) trabajamos enteramente en **Data Understanding**: no limpiamos de forma sistematica, no graficamos y no modelamos.

| Fase CRISP-DM | Pregunta guia | Donde aparece en el curso |
|---|---|---|
| Business Understanding | Que problema de negocio resolvemos? | S01: problema, KPI, Canvas v1 |
| **Data Understanding** | Que datos tenemos y que podemos confiar? | **S02-S04: EDA, DAMA-5, PECO y visualizacion** |
| Data Preparation | Como dejamos los datos listos? | S03: dataset preparado |
| Modeling | Que tecnica responde la pregunta? | S05-S10 |
| Evaluation | Sirve para tomar una decision? | S05-S10 |
| Deployment | Como se entrega y se mantiene? | S11 |

## 2.3.- Cuatro lentes para S02

Cada lente se nombra con su palabra ancla y se aplica a un fragmento concreto del dataset.

| Marco | Pregunta que responde | Palabras ancla | Donde se ve hoy |
|---|---|---|---|
| CRISP-DM | Donde estamos en el proyecto? | Business, Data Understanding, Preparation, Modeling, Evaluation, Deployment | S02 = Data Understanding |
| EDA | Que contiene la tabla? | forma, unidad de observacion, objetivo, anomalias, variables que faltan | `shape`, `head`, `info`, objetivo `No-show` |
| DAMA-5 | Puedo confiar en estos datos? | completitud, validez, consistencia, exactitud, unicidad | `validez`: edad negativa, espera negativa; `consistencia`: `No-show` invertido |
| PECO | Que comparacion quiero evaluar? | Population, Exposure, Comparison, Outcome | H1 espera, H2 dia, H3 SMS |

Pandas no es el objetivo: es la herramienta que ejecuta este razonamiento.

## 2.4.- Mapa de la sesion S02

| Paso | Accion | Lente aplicado |
|---:|---|---|
| 1 | Cargar el dataset (B1) | `Metodos Pandas: lectura de datos` |
| 2 | Inspeccionar la tabla (B2) | `EDA: forma + unidad + tipos` |
| 3 | Entender la variable objetivo (B3) | `EDA: objetivo` + `DAMA-5: consistencia` |
| 4 | Seleccionar y filtrar (B4) | `DAMA-5: validez` |
| 5 | Fechas y variables derivadas (B5) | `DAMA-5: validez` (temporal) |
| 6 | Dataset minimo de analisis (B6) | `DAMA-5: validez + consistencia` |
| 7 | Primeras evidencias H1/H2/H3 (B7) | `PECO: P/E/C/O observable` |
| 8 | Canvas v2 (B8) | `CRISP-DM: Data Understanding documentado` |

# 3.- B1 Cargar datos

> Lente principal: `Metodos Pandas: lectura de datos`.

Pandas convierte cualquier fuente externa en un `DataFrame`. Hoy usaremos solo `pd.read_csv`; el resto de la familia `pd.read_*` (Excel, JSON, Parquet, SQL) sigue exactamente el mismo patron de uso.

## 3.1.- Ficha: `Metodos Pandas: lectura de datos`

| Metodo | Para que sirve | Valor analitico hoy |
|---|---|---|
| `pd.read_csv()` | Archivos CSV | **No-Show Appointments** |
| `pd.read_excel()` | Excel | no usado |
| `pd.read_json()` | JSON | no usado |
| `pd.read_parquet()` | Parquet | no usado |
| `pd.read_sql()` | SQL | no usado |

**Validacion minima tras cargar:** que `df.shape` confirme el numero de filas que esperabamos (~110.527).

In [6]:
#  Imports y configuración
import pandas as pd
from google.colab import drive

#  Montar Drive y cargar dataset
drive.mount('/content/drive')
df = pd.read_csv(f"/content/drive/MyDrive/Classroom/KaggleV2-May-2016.csv")


print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
df.head(3)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset cargado: 110,527 filas × 14 columnas


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No


# 4.- B2 Inspeccionar la tabla

> Lente principal: `EDA: forma + unidad de observacion + tipos`.

Recorremos las primeras preguntas canonicas de EDA sobre la tabla cruda: que tamano tiene, como se ven los datos, que representa una fila y como estan tipadas las columnas.

## 4.1.- Ficha: `Metodos Pandas: primera mirada EDA`

| Metodo / atributo | Que hace | Pregunta EDA que responde |
|---|---|---|
| `df.shape` | Filas y columnas | EDA-1: que tamano tiene la tabla? |
| `df.head()` | Primeras 5 filas | EDA-2: como se ven los datos? |
| `df.sample()` | Filas aleatorias | EDA-2: que ejemplos reales hay? |
| `df.info()` | Tipos y nulos | EDA-1: que estructura tecnica tiene? |
| `df.describe()` | Estadisticas basicas | EDA-1: que rangos tienen las variables? |
| `df.dtypes` | Tipo de cada columna | EDA-1: como interpreta Pandas cada dato? |

## 4.2.- EDA-1 Forma

In [16]:
df.shape

(110527, 14)

**Lectura:** `(110527, 14)`. Las 110.527 filas son las que dieron el baseline 20.2% en S01.

## 4.3.- EDA-2 Primeras filas

In [17]:
df.head()

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


In [18]:
df.tail()

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
110522,2.572134e+12,5651768,F,2016-05-03T09:15:35Z,2016-06-07T00:00:00Z,56,MARIA ORTIZ,0,0,0,0,0,1,No
110523,3.596266e+12,5650093,F,2016-05-03T07:27:33Z,2016-06-07T00:00:00Z,51,MARIA ORTIZ,0,0,0,0,0,1,No
110524,1.557663e+13,5630692,F,2016-04-27T16:03:52Z,2016-06-07T00:00:00Z,21,MARIA ORTIZ,0,0,0,0,0,1,No
110525,9.213493e+13,5630323,F,2016-04-27T15:09:23Z,2016-06-07T00:00:00Z,38,MARIA ORTIZ,0,0,0,0,0,1,No
110526,3.775115e+14,5629448,F,2016-04-27T13:30:56Z,2016-06-07T00:00:00Z,54,MARIA ORTIZ,0,0,0,0,0,1,No


Columnas relevantes para el problema: `Age`, `ScheduledDay`, `AppointmentDay`, `SMS_received`, `No-show`.

## 4.4.- Sample reproducible (`random_state=14`)

In [46]:
df.sample(5, random_state=3)

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
54332,9.723352e+12,5670693,F,2016-05-06T14:26:24Z,2016-05-12T00:00:00Z,62,ILHA DO PRÍNCIPE,0,1,0,0,0,0,No
86406,4.644431e+13,5737460,M,2016-05-25T09:37:22Z,2016-06-01T00:00:00Z,12,BENTO FERREIRA,0,0,0,0,0,1,No
11303,8.895974e+11,5568234,F,2016-04-11T14:09:41Z,2016-05-09T00:00:00Z,72,JUCUTUQUARA,0,0,0,0,0,0,No
69839,7.244332e+14,5635834,F,2016-04-28T14:10:40Z,2016-05-31T00:00:00Z,78,CRUZAMENTO,0,1,1,0,0,0,No
20240,8.636298e+14,5683238,F,2016-05-11T07:21:22Z,2016-05-11T00:00:00Z,69,SANTO ANDRÉ,0,1,0,0,0,0,No


Convencion del curso: cualquier muestreo o split usa `random_state=14`. Garantiza reproducibilidad y comparacion entre estudiantes.

## 4.5.- EDA-2 Unidad de observacion

Pregunta clave: **que representa una fila?**

> Una fila = una cita, no un paciente. El mismo `PatientId` puede aparecer varias veces si tuvo varias citas.

Implicacion: si agrupasemos por paciente vs por cita, las tasas cambiarian.

## 4.6.- Estructura tecnica: `info` y `dtypes`

In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PatientId       110527 non-null  float64
 1   AppointmentID   110527 non-null  int64  
 2   Gender          110527 non-null  object 
 3   ScheduledDay    110527 non-null  object 
 4   AppointmentDay  110527 non-null  object 
 5   Age             110527 non-null  int64  
 6   Neighbourhood   110527 non-null  object 
 7   Scholarship     110527 non-null  int64  
 8   Hipertension    110527 non-null  int64  
 9   Diabetes        110527 non-null  int64  
 10  Alcoholism      110527 non-null  int64  
 11  Handcap         110527 non-null  int64  
 12  SMS_received    110527 non-null  int64  
 13  No-show         110527 non-null  object 
dtypes: float64(1), int64(8), object(5)
memory usage: 11.8+ MB


In [48]:
df.dtypes

,0
PatientId,float64
AppointmentID,int64
Gender,object
ScheduledDay,object
AppointmentDay,object
Age,int64
Neighbourhood,object
Scholarship,int64
Hipertension,int64
Diabetes,int64


**Lectura clave:** `ScheduledDay` y `AppointmentDay` aparecen como `object` (texto), no como fechas. Esto motiva la conversion del Bloque 5.

## 4.7.- Rangos numericos con `describe()`

`df.describe()` resume las variables numericas. No sustituye a la inspeccion detallada, pero ayuda a detectar valores extremos antes de interpretarlos.

In [49]:
df.describe()

,PatientId,AppointmentID,Age,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received
count,1.105270e+05,1.105270e+05,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000
mean,1.474963e+14,5.675305e+06,37.088874,0.098266,0.197246,0.071865,0.030400,0.022248,0.321026
std,2.560949e+14,7.129575e+04,23.110205,0.297675,0.397921,0.258265,0.171686,0.161543,0.466873
min,3.921784e+04,5.030230e+06,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.172614e+12,5.640286e+06,18.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,3.173184e+13,5.680573e+06,37.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,9.439172e+13,5.725524e+06,55.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,9.999816e+14,5.790484e+06,115.000000,1.000000,1.000000,1.000000,1.000000,4.000000,1.000000


**Lectura:** `Age` muestra un minimo de `-1`, que es imposible en el dominio. No lo corregimos todavia: lo etiquetamos como `DAMA-5: validez` y lo inspeccionamos en el Bloque 4.

# 5.- B3 Series y variable objetivo

> Lente principal: `EDA: objetivo` + `DAMA-5: consistencia`.

Entendemos la variable objetivo `No-show`: su distribucion, la tasa base y por que su codificacion (`Yes` = falto) induce a error. Luego construimos `NoShow` 0/1 en tres pasos explicitos.

## 5.1.- Ficha: `Metodos Pandas: columnas, conteos y tasas`

| Metodo | Que hace | Uso en el caso |
|---|---|---|
| `df["columna"]` | Selecciona una columna como `Series` | Extraer `No-show` |
| `.value_counts()` | Cuenta valores unicos | Distribucion de No-show |
| `.value_counts(normalize=True)` | Proporciones | Tasa base |
| `.mean()` sobre 0/1 | Promedio = tasa | Tasa de la nueva `NoShow` |
| `.astype()` | Convierte tipos | Pasar booleano a entero |

## 5.2.- Seleccionar columna como `Series`

In [51]:
df.head()

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


In [60]:
lista_ejemplo = [1, "a", 3, True, df["No-show"]]
print(lista_ejemplo)

[1, 'a', 3, True, 0         No
1         No
2         No
3         No
4         No
          ..
110522    No
110523    No
110524    No
110525    No
110526    No
Name: No-show, Length: 110527, dtype: object]


In [69]:
lista_ejemplo[0]

1

In [70]:
no_show_col = df["No-show"]
no_show_col.head()

,No-show
0,No
1,No
2,No
3,No
4,No


Una columna sola es una `Series`: igual que el DataFrame pero con una sola dimension. Tiene atributos (`.dtype`, `.name`) y metodos (`.head()`, `.value_counts()`).

## 5.3.- Distribucion de la variable objetivo

In [71]:
no_show_col.value_counts()

,count
No-show,
No,88208
Yes,22319


Salida esperada: 88.208 `No` y 22.319 `Yes`. El 22.319 es el numerador de la tasa base que vimos en S01.

## 5.4.- Tasa base (`normalize=True`)

In [79]:
no_show_col.value_counts(normalize=True)

,proportion
No-show,
No,0.798067
Yes,0.201933


Recuperamos el baseline: ~79.8% `No`, ~20.2% `Yes`.

## 5.5.- `DAMA-5: consistencia` - codificacion invertida

| Valor en la columna | Significado |
|---|---|
| `"Yes"` | El paciente **NO** asistio |
| `"No"` | El paciente **SI** asistio |

Riesgo: cualquier interpretacion ingenua de `Yes` como `asistio` se invierte. Solucion: construir `NoShow` 0/1 con semantica directa.

## 5.6.- Construir `NoShow` paso 1 - mascara booleana

In [82]:
df["No-show"].head(8)

,No-show
0,No
1,No
2,No
3,No
4,No
5,No
6,Yes
7,Yes


In [90]:
no_show_bool = df["No-show"] == "Yes"
no_show_bool.tail()

,No-show
110522,False
110523,False
110524,False
110525,False
110526,False


Una **mascara booleana** es el resultado de aplicar una condicion a una columna. Devuelve una `Series` de `True`/`False` con la misma longitud que la columna original. Es uno de los patrones mas usados en Pandas.

## 5.7.- Construir `NoShow` paso 2 - convertir a 0/1

In [103]:
df.head(1)

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No


In [ ]:
df["No-show"] = no_show_bool.astype(int)

In [104]:
df["NoShow"] = no_show_bool.astype(int)
df["NoShow"].head()

,NoShow
0,0
1,0
2,0
3,0
4,0


In [106]:
df.head(7)

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show,NoShow
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No,0
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No,0
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No,0
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No,0
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No,0
5,9.598513e+13,5626772,F,2016-04-27T08:36:51Z,2016-04-29T00:00:00Z,76,REPÚBLICA,0,1,0,0,0,0,No,0
6,7.336882e+14,5630279,F,2016-04-27T15:05:12Z,2016-04-29T00:00:00Z,23,GOIABEIRAS,0,0,0,0,0,0,Yes,1


`.astype(int)` convierte `True` a 1 y `False` a 0. Asignamos el resultado como nueva columna del DataFrame.

*Nota opcional:* `.astype("int8")` haria lo mismo ahorrando memoria. En 110.527 filas no se nota; en datasets de millones de filas si.

## 5.8.- Construir `NoShow` paso 3 - calcular la tasa

In [130]:
df["NoShow"].mean()

np.float64(0.20193255946510807)

**Lectura clave:** cuando una variable es 0/1, **su media es la tasa**. Hemos recuperado el 20.2% pero ahora con una variable que se lee correctamente.

*Version compacta (solo como referencia, no la usamos aqui):* `df["NoShow"] = (df["No-show"] == "Yes").astype(int)`.

# 6.- B4 Seleccionar y filtrar

> Lente principal: `DAMA-5: validez`.

Introducimos mascaras booleanas y `.loc` aplicandolos a la deteccion de un valor imposible (edad negativa). Es la primera vez que la tecnica de Pandas atiende directamente a una pregunta de calidad de datos.

In [99]:
df.columns

Index(['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay',
       'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hipertension',
       'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'No-show'],
      dtype='object')

## 6.1.- Ficha: `Metodos Pandas: seleccion y filtros`

| Patron | Que hace | Ejemplo |
|---|---|---|
| `df["col"]` | Selecciona una columna | `df["Age"]` |
| `df[["col1", "col2"]]` | Varias columnas | `df[["Age", "Gender"]]` |
| `df["col"] > valor` | Crea mascara booleana | `df["Age"] < 0` |
| `df.loc[filas, columnas]` | Seleccion por condicion | `df.loc[mask, ["Age"]]` |
| `df.iloc[]` | Seleccion por posicion | `df.iloc[0:5, 0:3]` |

In [107]:
df[["Age", "Gender", "SMS_received", "No-show", "NoShow"]].head()

,Age,Gender,SMS_received,No-show,NoShow
0,62,F,0,No,0
1,56,M,0,No,0
2,62,F,0,No,0
3,8,F,0,No,0
4,56,F,0,No,0


## 6.2.- EDA-4 Anomalias: edades negativas (`DAMA-5: validez`)

In [108]:
df.describe()

,PatientId,AppointmentID,Age,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,NoShow
count,1.105270e+05,1.105270e+05,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000
mean,1.474963e+14,5.675305e+06,37.088874,0.098266,0.197246,0.071865,0.030400,0.022248,0.321026,0.201933
std,2.560949e+14,7.129575e+04,23.110205,0.297675,0.397921,0.258265,0.171686,0.161543,0.466873,0.401444
min,3.921784e+04,5.030230e+06,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.172614e+12,5.640286e+06,18.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,3.173184e+13,5.680573e+06,37.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,9.439172e+13,5.725524e+06,55.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,9.999816e+14,5.790484e+06,115.000000,1.000000,1.000000,1.000000,1.000000,4.000000,1.000000,1.000000


In [110]:
age_negative_mask = df["Age"] < 0
age_negative_mask.sum()
#age_negative_mask

np.int64(1)

**Lectura:** 1 fila con edad negativa. Una mascara es una `Series` de True/False; en Python `True` vale 1 y `False` vale 0, asi que `.sum()` cuenta cuantas filas cumplen la condicion. La etiqueta `DAMA-5: validez` nombra el problema: una edad negativa es imposible en el dominio.

## 6.3.- Inspeccionar la fila problematica con `.loc`

In [121]:
df.loc[:10, ["Age", "Gender", "No-show"]]

,Age,Gender,No-show
0,62,F,No
1,56,M,No
2,62,F,No
3,8,F,No
4,56,F,No
5,76,F,No
6,23,F,Yes
7,39,F,Yes
8,21,F,No
9,19,F,No


In [123]:
df.columns

Index(['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay',
       'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hipertension',
       'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'No-show',
       'NoShow'],
      dtype='object')

In [124]:
df.loc[age_negative_mask, ["PatientId", "Age", "Gender", "No-show"]]

,PatientId,Age,Gender,No-show
99832,4.659432e+14,-1,F,No


Salida esperada: una sola fila con `Age = -1`. Probablemente un error de carga. La decision de excluirla se tomara en S03 (limpieza sistematica); hoy solo la identificamos y la dejaremos fuera al construir `df_analysis` en el Bloque 6.

## 6.4.- `.loc` vs `.iloc`

In [126]:
df.head(1)

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show,NoShow
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No,0


In [129]:
df.iloc[8]

,8
PatientId,56394729949972.0
AppointmentID,5638447
Gender,F
ScheduledDay,2016-04-29T08:02:16Z
AppointmentDay,2016-04-29T00:00:00Z
Age,21
Neighbourhood,ANDORINHAS
Scholarship,0
Hipertension,0
Diabetes,0


- `.loc` indexa por **nombre** de fila/columna o **mascara**. Es el patron por defecto.
- `.iloc` indexa por **posicion** (entero). Util cuando solo importa el orden.

En el trabajo diario `.loc` domina.

## 6.5.- Descanso (20:00-20:20)

20 minutos. Al volver retomamos con un prompt clinic antes de pasar a fechas.

## 6.6.- Prompt clinic 1 - filtro sin inventar columnas

El estudiante acaba de ver su primera mascara booleana + `.loc`. Momento natural para practicar como pedirle a un LLM ayuda con filtros similares sin que invente esquemas.

**Prompt malo**

```text
Hazme un filtro de edad invalida en mi dataset.
```

**Prompt bueno** (aplica checklist S01 puntos 3 y 7; ver `S01_2604_guia_clase.md` sec 4.3)

```text
Tengo un DataFrame `df` con citas medicas. La columna "Age" es entero
y los valores < 0 son invalidos (errores de carga). Genera codigo Pandas
que cuente cuantas filas tienen edad invalida y cuantas no, usando una
mascara booleana. No inventes columnas que no te he descrito.
```

**Diferencia esperada:** el prompt malo suele inventar `patient_age` o `birth_year`; el prompt bueno usa `Age` y devuelve un conteo verificable.

Microreto: reformula en 90 segundos un prompt para tu propio caso de filtro.

In [132]:
# Crear máscara booleana para edades inválidas
invalid_age_mask = df["Age"] < 0

# Contar cuántas filas son inválidas y cuántas no
print(f"Filas con edad inválida: {invalid_age_mask.sum()}")
print(f"Filas con edad válida: {(~invalid_age_mask).sum()}")

# Mostrar los registros inválidos para inspección
display(df.loc[invalid_age_mask])

Filas con edad inválida: 1
Filas con edad válida: 110526


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show,NoShow
99832,4.659432e+14,5775010,F,2016-06-06T08:58:13Z,2016-06-06T00:00:00Z,-1,ROMÃO,0,0,0,0,0,0,No,0


# 7.- B5 Fechas y variables derivadas

> Lente principal: `DAMA-5: validez` (temporal) + `EDA: variables faltantes`.

Convertimos texto a fecha, normalizamos, construimos `WaitingDays` y `AppointmentWeekday`. Es el bloque con mas momentos de comprension; sin entender por que normalizar, el resultado de `WaitingDays` parece magia.

## 7.1.- Ficha: `Metodos Pandas: fechas`

| Metodo / accesor | Que hace | Uso en el caso |
|---|---|---|
| `pd.to_datetime()` | Convierte texto a fecha | `ScheduledDay`, `AppointmentDay` |
| `.dt` | Accede a partes temporales de una `Series` | Puente a las dos siguientes |
| `.dt.normalize()` | Quita la hora, conserva fecha | Comparar dias calendario |
| `.dt.days` | Extrae dias de un `timedelta` | Crear `WaitingDays` |
| `.dt.day_name()` | Extrae nombre del dia | Crear `AppointmentWeekday` |

## 7.2.- Por que normalizar las fechas

Contraste visual:

```text
Sin normalizar:
ScheduledDay:   2016-04-29 18:38:00
AppointmentDay: 2016-04-29 00:00:00
Diferencia: -18 horas (parece negativo!)

Con .dt.normalize():
ScheduledDay:   2016-04-29 00:00:00
AppointmentDay: 2016-04-29 00:00:00
Diferencia: 0 dias (correcto)
```

> Normalizar **reduce falsos negativos por hora**, pero no elimina inconsistencias reales de fecha.

## 7.3.- Convertir fechas paso 1 - a `datetime64`

In [137]:
df["ScheduledDay"]

,ScheduledDay
0,2016-04-29T18:38:08Z
1,2016-04-29T16:08:27Z
2,2016-04-29T16:19:04Z
3,2016-04-29T17:29:31Z
4,2016-04-29T16:07:23Z
...,...
110522,2016-05-03T09:15:35Z
110523,2016-05-03T07:27:33Z
110524,2016-04-27T16:03:52Z
110525,2016-04-27T15:09:23Z


In [139]:
scheduled = pd.to_datetime(df["ScheduledDay"])
scheduled.head()

,ScheduledDay
0,2016-04-29 18:38:08+00:00
1,2016-04-29 16:08:27+00:00
2,2016-04-29 16:19:04+00:00
3,2016-04-29 17:29:31+00:00
4,2016-04-29 16:07:23+00:00


In [133]:
scheduled = pd.to_datetime(df["ScheduledDay"])
appointment = pd.to_datetime(df["AppointmentDay"])
scheduled.head()

,ScheduledDay
0,2016-04-29 18:38:08+00:00
1,2016-04-29 16:08:27+00:00
2,2016-04-29 16:19:04+00:00
3,2016-04-29 17:29:31+00:00
4,2016-04-29 16:07:23+00:00


In [134]:
appointment.head()

,AppointmentDay
0,2016-04-29 00:00:00+00:00
1,2016-04-29 00:00:00+00:00
2,2016-04-29 00:00:00+00:00
3,2016-04-29 00:00:00+00:00
4,2016-04-29 00:00:00+00:00


Las fechas dejan de ser texto (`object`) y pasan a tipo `datetime64`, sobre el que ya se puede aplicar aritmetica temporal.

## 7.4.- Convertir fechas paso 2 - normalizar

In [142]:
df["ScheduledDay"].head(2)

,ScheduledDay
0,2016-04-29T18:38:08Z
1,2016-04-29T16:08:27Z


In [140]:
scheduled = pd.to_datetime(df["ScheduledDay"])
scheduled.head(2)

,ScheduledDay
0,2016-04-29 18:38:08+00:00
1,2016-04-29 16:08:27+00:00


In [135]:
scheduled_date = scheduled.dt.normalize()
appointment_date = appointment.dt.normalize()

In [141]:
scheduled_date.head(2)

,ScheduledDay
0,2016-04-29 00:00:00+00:00
1,2016-04-29 00:00:00+00:00


Ambas `Series` ahora tienen hora 00:00:00. La diferencia entre ellas dara dias calendario completos.

## 7.5.- Convertir fechas paso 3 - calcular `WaitingDays`

In [144]:
df.columns

Index(['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay',
       'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hipertension',
       'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'No-show',
       'NoShow'],
      dtype='object')

In [149]:
df[["AppointmentDay", "ScheduledDay"]].head()

,AppointmentDay,ScheduledDay
0,2016-04-29T00:00:00Z,2016-04-29T18:38:08Z
1,2016-04-29T00:00:00Z,2016-04-29T16:08:27Z
2,2016-04-29T00:00:00Z,2016-04-29T16:19:04Z
3,2016-04-29T00:00:00Z,2016-04-29T17:29:31Z
4,2016-04-29T00:00:00Z,2016-04-29T16:07:23Z


In [143]:
waiting_time = appointment_date - scheduled_date
waiting_time

,0
0,0 days
1,0 days
2,0 days
3,0 days
4,0 days
...,...
110522,35 days
110523,35 days
110524,41 days
110525,41 days


In [151]:
waiting_time = appointment_date - scheduled_date
df["WaitingDays"] = waiting_time.dt.days
df["WaitingDays"].describe()

,WaitingDays
count,110527.000000
mean,10.183702
std,15.254996
min,-6.000000
25%,0.000000
50%,4.000000
75%,15.000000
max,179.000000


**Lectura:** `min = -6`, `max = 179`, `mean ~ 10`. Aun quedan 5 esperas negativas despues de normalizar. No son errores horarios (esos los corrige `.dt.normalize()`), sino inconsistencias reales del dataset.

*Comparacion:* sin normalizar habia 38.568 valores negativos. Con `normalize` quedan 5.

`DAMA-5: validez` ampliado: edad negativa (1) + espera negativa (5).

## 7.6.- Crear `AppointmentWeekday`

In [154]:
df["AppointmentWeekday"] = appointment.dt.day_name()
df["AppointmentWeekday"].tail(10)

,AppointmentWeekday
110517,Tuesday
110518,Tuesday
110519,Tuesday
110520,Tuesday
110521,Tuesday
110522,Tuesday
110523,Tuesday
110524,Tuesday
110525,Tuesday
110526,Tuesday


Nombres de dia en ingles (`Monday`, `Tuesday`, ...). Lo usaremos para H2 en el Bloque 7.

In [164]:
df.columns

Index(['PatientId', 'AppointmentID', 'Gender', 'ScheduledDay',
       'AppointmentDay', 'Age', 'Neighbourhood', 'Scholarship', 'Hipertension',
       'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'No-show',
       'NoShow', 'WaitingDays', 'AppointmentWeekday'],
      dtype='object')

In [167]:
df.loc[4:10,["AppointmentDay", "ScheduledDay", "WaitingDays", "AppointmentWeekday"]]

,AppointmentDay,ScheduledDay,WaitingDays,AppointmentWeekday
4,2016-04-29T00:00:00Z,2016-04-29T16:07:23Z,0,Friday
5,2016-04-29T00:00:00Z,2016-04-27T08:36:51Z,2,Friday
6,2016-04-29T00:00:00Z,2016-04-27T15:05:12Z,2,Friday
7,2016-04-29T00:00:00Z,2016-04-27T15:39:58Z,2,Friday
8,2016-04-29T00:00:00Z,2016-04-29T08:02:16Z,0,Friday
9,2016-04-29T00:00:00Z,2016-04-27T12:48:25Z,2,Friday
10,2016-04-29T00:00:00Z,2016-04-27T14:58:11Z,2,Friday


# 8.- B6 Dataset minimo de analisis (`df_analysis`)

> Lente principal: `DAMA-5: validez + consistencia`.

Combinamos dos mascaras con `&` para producir `df_analysis`, la base sobre la que evaluaremos H1, H2 y H3. Esto no es limpieza sistematica (eso es S03); es validacion minima para no interpretar basura.

## 8.1.- Ficha: `Metodos Pandas: combinacion de mascaras`

| Patron | Que hace |
|---|---|
| `mask1 & mask2` | Combina mascaras tipo AND |
| `mask1 \| mask2` | Combina mascaras tipo OR |
| `.copy()` | Crea una copia explicita del subconjunto |
| `len(df)` | Cuenta filas |
| `.sum()` sobre booleanos | Cuenta valores `True` |

## 8.2.- Construir `df_analysis` paso a paso

In [173]:
valid_age = df["Age"] >= 0
valid_waiting = df["WaitingDays"] >= 0
valid_rows = valid_age & valid_waiting

df_analysis = df.loc[valid_rows].copy()
len(df), len(df_analysis)

(110527, 110521)

In [172]:
df.shape

(110527, 17)

In [171]:
valid_rows.head()

,0
0,True
1,True
2,True
3,True
4,True


In [ ]:
df_analysis

Salida esperada: `(110527, 110521)`. Retiramos 6 filas: 1 edad negativa + 5 esperas negativas.

**Nota tecnica sobre `&`:** para combinar dos mascaras booleanas usamos `&`, no `and`. Cuando las mascaras son **variables** (como aqui), no necesitamos parentesis. Si las escribieramos como expresiones inline (`(df["Age"] >= 0) & (df["WaitingDays"] >= 0)`), los parentesis serian obligatorios por precedencia de operadores.

## 8.3.- Validacion minima != limpieza sistematica

> Hoy quitamos solo registros claramente invalidos para no contaminar las primeras interpretaciones. En S03 abordaremos la preparacion sistematica: renombrar columnas, gestionar duplicados, decidir imputaciones, encapsular todo en `prepare(df)`. `df_analysis` es la base de hoy; no es el dataset final.

# 9.- B7 Primeras evidencias H1, H2, H3

> Lente principal: `PECO: P/E/C/O observable`.

Convertimos las tres hipotesis del Canvas v1 en evidencia preliminar. Antes de tocar `groupby`, fijamos la matriz PECO conjunta para ver que cambia entre hipotesis.

## 9.1.- Ficha: `Metodos Pandas: resumen por grupos`

| Metodo | Que hace | Pregunta |
|---|---|---|
| `.mean()` sobre 0/1 | Calcula la proporcion | Cual es la tasa? |
| `.groupby()` | Agrupa filas por categoria | Cambia por segmento? |
| `.agg(["mean", "count"])` | Varias metricas a la vez | Tasa y cuantos casos? |
| `.sort_values()` | Ordena resultados | Donde es mayor o menor? |

**Regla del bloque:** en cualquier agrupacion, mostrar siempre `mean` y `count` juntos.

## 9.2.- Matriz PECO conjunta H1 / H2 / H3

| Hipotesis | P | E | C | O |
|---|---|---|---|---|
| H1: espera larga | citas validas en `df_analysis` | `WaitingDays > 14` | `WaitingDays <= 14` | `NoShow` |
| H2: dia de semana | citas validas en `df_analysis` | un valor de `AppointmentWeekday` | otros dias / resto | `NoShow` |
| H3: SMS | citas validas en `df_analysis` | `SMS_received == 1` | `SMS_received == 0` | `NoShow` |

**Population** y **Outcome** se mantienen casi constantes. Lo que cambia entre hipotesis es la forma de definir **Exposure** y **Comparison**.

Regla practica: **P** = sobre quien calculo. **E** = grupo observado. **C** = contra quien comparo. **O** = resultado que resumo.

Cuidado: `E` no implica causalidad. En EDA es un **segmento observado**, no un tratamiento asignado.

## 9.3.- H1 PECO - afecta la espera al no-show?

Tomamos la fila H1 de la matriz PECO y la convertimos en dos mascaras.

In [ ]:
short_wait_mask = df_analysis["WaitingDays"] <= 14
long_wait_mask = df_analysis["WaitingDays"] > 14

short_wait_rate = df_analysis.loc[short_wait_mask, "NoShow"].mean()
long_wait_rate = df_analysis.loc[long_wait_mask, "NoShow"].mean()

short_wait_rate, long_wait_rate

(np.float64(0.15981249697965497), np.float64(0.327435222890915))

**Lectura de negocio:**

- Espera `<= 14` dias: tasa ~ 16.0%
- Espera `> 14` dias: tasa ~ 32.7%
- Diferencia: ~ +16.8 p.p.

> Cuando la espera supera dos semanas, la tasa de no-show practicamente se duplica. La espera larga aparece como palanca prioritaria para investigar en S03.

## 9.4.- H2 - influye el dia de la semana?

`groupby` implementa la comparacion PECO cuando la exposicion tiene categorias. Cada dia actua como **E** frente al resto.

In [ ]:
weekday_grouped = df_analysis.groupby("AppointmentWeekday")["NoShow"]
weekday_summary = weekday_grouped.agg(["mean", "count"])
weekday_summary = weekday_summary.sort_values("mean", ascending=False)
weekday_summary

,mean,count
AppointmentWeekday,,
Saturday,0.230769,39
Friday,0.212261,19019
Monday,0.206446,22713
Tuesday,0.200874,25638
Wednesday,0.196861,25866
Thursday,0.193494,17246


Salida esperada: tabla con 6 filas (dias con citas en el dataset) y columnas `mean` y `count`. La interpretacion concreta la hacemos en 9.6 (slide "Lectura: denominador y causalidad").

## 9.5.- H3 - el SMS reduce el no-show?

Comparacion binaria: citas con SMS registrado (`E`) vs citas sin SMS registrado (`C`).

In [ ]:
sms_grouped = df_analysis.groupby("SMS_received")["NoShow"]
sms_summary = sms_grouped.agg(["mean", "count"])
sms_summary

,mean,count
SMS_received,,
0,0.166980,75039
1,0.275745,35482


Salida esperada: dos filas (`0` y `1`) con sus tasas y conteos. La interpretacion (causalidad) la hacemos en 9.6.

## 9.6.- Lectura: denominador y causalidad

Slide de juicio analitico separada. Dos riesgos sobre los resultados de H2 y H3:

| Riesgo | Pregunta docente | Ejemplo en S02 |
|---|---|---|
| Tasa sin denominador | Cuantos casos sostienen esa tasa? | Sabado puede tener pocos casos: `count` bajo invalida cualquier decision operativa |
| Correlacion sin causalidad | El factor causa el outcome o solo lo marca? | Las citas con SMS pueden corresponder a pacientes que el sistema ya considera de riesgo (sesgo de seleccion) |

> Una tasa alta con pocos casos no equivale a una decision operativa. Y una asociacion observacional (`SMS` -> `NoShow`) no es causalidad. Para evaluar causalidad necesitariamos historico mas rico, asignacion controlada o ajuste estadistico (S03 / S05).

## 9.7.- Microejercicio H4 PECO - paso 1: completar la fila

Antes de programar, completa la fila PECO sobre papel/pizarra. Pista: queremos comparar `Age >= 60` frente a `Age < 60` y resumir `NoShow`.

| Hipotesis | P | E | C | O |
|---|---|---|---|---|
| H4: edad alta | citas validas en `df_analysis` | ? | ? | ? |

**Solucion esperada (no la mires hasta intentar):**

| Hipotesis | P | E | C | O |
|---|---|---|---|---|
| H4: edad alta | citas validas en `df_analysis` | `Age >= 60` | `Age < 60` | `NoShow` |

## 9.8.- Microejercicio H4 - paso 2: traducir la fila PECO a Pandas

Replica el patron de H1 sobre `df_analysis`. Solucion comentada abajo.

In [ ]:
# Tu codigo aqui (intenta antes de mirar la celda siguiente)


In [ ]:
# Solucion sugerida
older_appointments = df_analysis.loc[df_analysis["Age"] >= 60, "NoShow"].mean()
younger_appointments = df_analysis.loc[df_analysis["Age"] < 60, "NoShow"].mean()

older_appointments, younger_appointments

(np.float64(0.1530795390137918), np.float64(0.21346629509004017))

Pregunta de comparacion: la diferencia en p.p. entre `Age >= 60` y `Age < 60` es mayor o menor que la de H1 (`WaitingDays > 14` vs `<= 14`, que dio ~ +16.8 p.p.)?

En S03 generalizaremos este patron con `pd.cut` para crear `AgeGroup`.

## 9.9.- Prompt clinic 2 - pedir a un LLM que explique un error de Pandas

Momento: si en H1/H2/H3 ha aparecido un `SettingWithCopyWarning` o similar, es el material natural del clinic.

**Prompt malo**

```text
No me funciona el codigo, ayuda.
```

**Prompt bueno**

```text
Tengo este codigo Pandas en Python 3:

subset = df.loc[df["Age"] >= 0]
subset["WaitingDays"] = ...

Me da este error:
SettingWithCopyWarning: A value is trying to be set on a copy
of a slice from a DataFrame.

Mi pregunta: por que ocurre este warning y como lo corrijo sin
cambiar la logica? Usa el mismo nombre de columna en tu respuesta.
```

**Diferencia esperada:** el prompt malo obliga al LLM a adivinar; el bueno entrega contexto completo y recibe una explicacion accionable con el mismo nombre de variable.

Checklist S01 implicado: puntos 1 (contexto), 4 (resultado esperado) y 5 (peticion de explicacion). Ver `S01_2604_guia_clase.md` sec 4.3.

# 10.- B8 Canvas v2 y cierre

> Lente principal: `CRISP-DM: Data Understanding documentado`.

Cerramos la fase actualizando el Project Canvas con los conteos reales y formulamos el puente a S03.

## 10.1.- Actualizar el Canvas en clase

| Casilla | Que actualizamos |
|---|---|
| **3 - Datos** | 110.527 brutos / 110.521 validos (`df_analysis`) / variables derivadas `WaitingDays`, `NoShow`, `AppointmentWeekday` |
| **4 - DAMA-5** | `validez`: edad negativa (1) y espera negativa (5). `consistencia`: `No-show` invertido |
| **5 - PECO** | H1: ~ +16.8 p.p. / H2: mirar `count` antes de decidir / H3: cautela con causalidad |
| **6 - Decision** | Pendiente para S03 (consolidar tras preparacion sistematica) |

Tarea individual: completa el borrador editable de este notebook y despues traslada la version final a [`../S02_2604_canvas_update.md`](../S02_2604_canvas_update.md).

## 10.2.- Del Canvas v1 al Canvas v2

El Canvas no se rehace desde cero. En S02 refinamos el v1 con evidencia tabular real obtenida durante la clase.

| Casilla | Estado tras S01 | Que aporta S02 |
|---|---|---|
| 1 - Problema | frase de negocio inicial | revisar solo si la evidencia cambia el alcance |
| 2 - KPI | baseline 20.2% | confirmado con `NoShow` 0/1 |
| 3 - Datos | descripcion general del CSV | filas brutas, filas validas y variables derivadas |
| 4 - DAMA-5 | riesgos detectados de forma inicial | riesgos cuantificados y etiquetados |
| 5 - PECO | hipotesis sin verificar | primeras tasas, diferencias y denominadores |
| 6 - Decision | decision tentativa | mantener con limite explicito hasta S03 |

Regla practica para el cierre: **una casilla, una decision escrita, una cifra cuando exista**.

## 10.3.- Helper: cifras clave para el Canvas v2

Ejecuta la siguiente celda para tener a mano las cifras del dia. No introduce conceptos nuevos: solo reune los resultados que ya calculamos durante la sesion.

In [ ]:
# Canvas v2 helper: key figures from today's analysis.
# These values come from variables already created in the notebook.

print("CASILLA 3 - DATOS")
print(f"  Filas brutas:   {len(df)}")
print(f"  Filas validas:  {len(df_analysis)}  (df_analysis)")
print(f"  Retiradas:      {len(df) - len(df_analysis)}  (edad < 0 o espera < 0)")
print(f"  Tasa baseline:  {df_analysis['NoShow'].mean():.3%}")
print()

print("CASILLA 4 - DAMA-5")
print(f"  validez (Age < 0):           {(df['Age'] < 0).sum()} fila")
print(f"  validez (WaitingDays < 0):   {(df['WaitingDays'] < 0).sum()} filas (tras normalize)")
print("  consistencia: 'No-show' invertido -> resuelto creando NoShow 0/1")
print()

print("CASILLA 5 - PECO H1 (espera)")
print(f"  Tasa espera <= 14:  {short_wait_rate:.3%}")
print(f"  Tasa espera >  14:  {long_wait_rate:.3%}")
print(f"  Diferencia:         {(long_wait_rate - short_wait_rate) * 100:+.1f} p.p.")
print()

print("CASILLA 5 - PECO H2 (top y bottom dia)")
print("  Top:")
print(weekday_summary.head(1).to_string())
print("  Bottom:")
print(weekday_summary.tail(1).to_string())
print()

print("CASILLA 5 - PECO H3 (SMS)")
print(sms_summary.to_string())

CASILLA 3 - DATOS
  Filas brutas:   110527
  Filas validas:  110521  (df_analysis)
  Retiradas:      6  (edad < 0 o espera < 0)
  Tasa baseline:  20.190%

CASILLA 4 - DAMA-5
  validez (Age < 0):           1 fila
  validez (WaitingDays < 0):   5 filas (tras normalize)
  consistencia: 'No-show' invertido -> resuelto creando NoShow 0/1

CASILLA 5 - PECO H1 (espera)
  Tasa espera <= 14:  15.981%
  Tasa espera >  14:  32.744%
  Diferencia:         +16.8 p.p.

CASILLA 5 - PECO H2 (top y bottom dia)
  Top:
                        mean  count
AppointmentWeekday                 
Saturday            0.230769     39
  Bottom:
                        mean  count
AppointmentWeekday                 
Thursday            0.193494  17246

CASILLA 5 - PECO H3 (SMS)
                  mean  count
SMS_received                 
0             0.166980  75039
1             0.275745  35482


## 10.4.- Borrador editable del Canvas v2

Completa las siguientes celdas en el propio notebook. Usa frases cortas, numeros reales y una lectura de negocio. Si una casilla no cambia respecto a tu v1, escribe `(sin cambios)` y justifica en media frase.

### 10.4.1.- Casilla 3 - Datos disponibles

**Pregunta guia:** que sabemos ahora del dataset que no podiamos afirmar al final de S01?

**Respuesta del equipo:**

- Filas brutas:
- Filas validas:
- Unidad de observacion:
- Variables derivadas creadas:
- Dato que pedirias al negocio:

### 10.4.2.- Casilla 4 - Riesgos DAMA-5

**Pregunta guia:** que riesgos ya no son intuiciones, sino hallazgos medidos?

**Respuesta del equipo:**

- Validez:
- Consistencia:
- Impacto sobre `df_analysis`:
- Riesgo que queda pendiente:

### 10.4.3.- Casilla 5 - Hipotesis PECO

**Pregunta guia:** que hipotesis quedan reforzadas, matizadas o abiertas tras mirar tasas y denominadores?

**Respuesta del equipo:**

- H1 espera:
- H2 dia de semana:
- H3 SMS:
- Limite de interpretacion:

### 10.4.4.- Casilla 6 - Decision candidata

**Pregunta guia:** que decision mantendrias de forma tentativa y que limite pondrias por escrito?

**Respuesta del equipo:**

- Decision candidata:
- Evidencia que la apoya hoy:
- Limite v2 hasta S03:

### 10.4.5.- Pregunta nueva para S03

**Pregunta guia:** que pregunta de negocio ha aparecido hoy y aun no puede responderse con este notebook?

**Respuesta del equipo:**

- Pregunta:
- Por que importa:
- Que variable, cruce o preparacion necesitarias:

## 10.5.- Pasar del borrador al markdown

Cuando termines el borrador, abre [`../S02_2604_canvas_update.md`](../S02_2604_canvas_update.md) y traslada tus respuestas a las casillas correspondientes.

Entrega esperada antes de S03: Canvas v2 con casillas 3, 4 y 5 actualizadas, casilla 6 con limite explicito y una pregunta nueva para la proxima clase.

## 10.6.- Puente a S03

> Todo lo que hoy hicimos a mano (limpiar minimamente, derivar, agrupar) en S03 lo encapsulamos en una funcion `prepare(df)` y profundizamos `pd.cut`, `pivot_table`, `merge` y `crosstab`. La proxima clase pasamos de "evidencias preliminares" a "dataset preparado y reproducible".

## 10.7.- Cierre

> Hoy hemos abierto el dataset a escala. Una fila es una cita. Son 110.527 brutos y 110.521 validos. Sabemos que tipos hay y donde fallan: la edad tiene 1 caso negativo y `WaitingDays` deja 5 esperas negativas tras normalizar fechas (`DAMA-5: validez`); la codificacion de `No-show` es invertida (`DAMA-5: consistencia`). Hemos construido `NoShow` paso a paso, derivado `WaitingDays` y `AppointmentWeekday`, generado `df_analysis` con 110.521 citas combinando dos mascaras con `&`. Hemos explorado preliminarmente H1, H2 y H3 con `groupby` + `agg(["mean", "count"])`, mostrando que el denominador importa y que el SMS puede tener sesgo de seleccion. El Canvas v2 queda orientado con cifras reales para las casillas 3, 4 y 5, y una decision todavia limitada hasta S03. La proxima clase encapsulamos todo esto en una funcion y profundizamos preparacion.